# Bank Marketing Exploratory Data Analysis & Deep Learning Classification
## Exploratory Data Analysis and Visualization - Assignment 2

---

### 1. Introduction and Objectives

#### 1.1 Project Overview
Direct marketing campaigns remain one of the most vital strategies employed by banking institutions to solicit term deposit subscriptions. A term deposit represents a fixed-term investment product where a customer commits a specific monetary balance to the bank for an agreed duration in exchange for a predetermined interest rate. For financial institutions, securing term deposits is essential because they provide predictable liquidity, expand operational lending capacity, and solidify long-term client relationships.

However, traditional mass telemarketing incurs substantial operational overhead, risks customer attrition through contact fatigue, and often yields low conversion rates. To overcome these inefficiencies, data-driven financial analytics leverages machine learning and deep learning to model customer demographic attributes, previous banking interactions, and economic indicators to predict the propensity of a client to subscribe to a term deposit.

#### 1.2 Core Objectives
This notebook executes a rigorous end-to-end data science and deep learning workflow structured across the following primary goals:
1. **Data Ingestion and Integrity Verification**: Seamlessly ingest the Bank Marketing dataset in Google Colab and local environments, inspect structural dimensions, attribute data types, and check for missing values or duplicates.
2. **Exploratory Data Analysis (EDA)**: Perform comprehensive univariate, bivariate, and multivariate statistical analyses to identify significant behavioral, demographic, and campaign-level drivers of customer conversion.
3. **Publication-Quality Visualization**: Design and render high-resolution visual analytics (using Matplotlib and Seaborn) and export publication-ready PNG graphics.
4. **Data Leakage Mitigation**: Address the well-documented domain nuance regarding call `duration`. While call duration is retained for diagnostic EDA, it is strictly excised from the predictive feature matrix prior to model development, ensuring authentic out-of-sample prospect evaluation.
5. **Data Preprocessing & Robust Partitioning**: Encode categorical predictors via one-hot encoding, standardize continuous features, and execute a stratified three-way split (70% Training, 15% Validation, 15% Test) to prevent data snooping.
6. **Deep Learning Model Development**: Architect, compile, regularize (via Dropout, Batch Normalization, and $L_2$ weight penalty), and train an Artificial Neural Network (ANN) using TensorFlow/Keras.
7. **Comprehensive Evaluation**: Generate diagnostic curves (loss curves, accuracy curves), Receiver Operating Characteristic (ROC) curves, Precision-Recall metrics, and Confusion Matrices to evaluate generalization performance.
8. **Feature Importance Interpretation**: Perform model-agnostic Permutation Feature Importance to extract actionable business insights for marketing optimization.


---
### 2. Dataset Loading

In this section, we set up our workspace directories, import standard scientific libraries, and establish a flexible data ingestion pipeline that automatically detects whether execution is occurring in **Google Colab** or a **local environment**.

The loader searches for `Dataset/bank-full.csv`, `Dataset/bank.csv`, or the root directory equivalent. If neither is detected when running in Google Colab, an interactive file upload prompt is initiated automatically.


In [ ]:
# System and Environment Setup
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data Manipulation and Linear Algebra
import numpy as np
import pandas as pd

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

# Scikit-Learn Preprocessing & Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_curve,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score
)

# Deep Learning Framework (TensorFlow / Keras)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks

# Set random seeds for absolute reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Ensure project directories exist
os.makedirs('Dataset', exist_ok=True)
os.makedirs('Notebook', exist_ok=True)
os.makedirs('Visualizations', exist_ok=True)
os.makedirs('Report', exist_ok=True)

print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version:      {np.__version__}")
print(f"Pandas Version:     {pd.__version__}")
print("Directory structure successfully initialized.")


In [ ]:
# Robust Dataset Loading Pipeline
def load_bank_dataset():
    """
    Searches for the Bank Marketing dataset across local and Google Colab paths.
    Supports both comma (,) and semicolon (;) delimited CSV files.
    Standardizes the target column name to 'deposit'.
    """
    candidate_paths = [
        'Dataset/bank-full.csv',
        'Dataset/bank.csv',
        'bank-full.csv',
        'bank.csv',
        '../Dataset/bank-full.csv',
        '../Dataset/bank.csv'
    ]
    
    file_path = None
    for path in candidate_paths:
        if os.path.exists(path):
            file_path = path
            break
            
    if file_path is None:
        # Check if running in Google Colab to prompt user upload
        try:
            from google.colab import files
            print("Dataset not found in local workspace. Please upload 'bank-full.csv' or 'bank.csv':")
            uploaded = files.upload()
            for filename in uploaded.keys():
                file_path = filename
                break
        except ImportError:
            raise FileNotFoundError(
                "Bank Marketing CSV dataset was not found. Please place 'bank-full.csv' or 'bank.csv' inside the 'Dataset/' folder."
            )
            
    print(f"Loading dataset from: {file_path}")
    
    # Detect delimiter automatically by inspecting the first line
    with open(file_path, 'r', encoding='utf-8') as f:
        first_line = f.readline()
        delimiter = ';' if ';' in first_line and ',' not in first_line else ','
        
    df = pd.read_csv(file_path, sep=delimiter)
    
    # Standardize target column: handle 'y' or 'deposit'
    if 'deposit' not in df.columns and 'y' in df.columns:
        df.rename(columns={'y': 'deposit'}, inplace=True)
        
    print(f"Dataset successfully loaded. Shape: {df.shape[0]:,} rows by {df.shape[1]} columns.")
    return df

# Load the actual dataset
df = load_bank_dataset()
df.head()


---
### 3. Data Understanding

In this section, we examine the structural properties of the dataset, document the domain definition of each attribute, verify data types, inspect memory utilization, and assess the target variable distribution.


#### Attribute Metadata and Domain Taxonomy

| # | Attribute Name | Data Type | Attribute Description & Domain Category |
|---|---|---|---|
| 1 | `age` | Numerical (int) | Age of the prospective client in years. |
| 2 | `job` | Categorical (obj) | Occupation type (`admin.`, `blue-collar`, `entrepreneur`, `housemaid`, `management`, `retired`, `self-employed`, `services`, `student`, `technician`, `unemployed`, `unknown`). |
| 3 | `marital` | Categorical (obj) | Marital status (`divorced`, `married`, `single`; note: `divorced` includes widowed). |
| 4 | `education` | Categorical (obj) | Highest level of education attained (`primary`, `secondary`, `tertiary`, `unknown`). |
| 5 | `default` | Categorical (obj) | Credit default flag: does the client currently have credit in default? (`no`, `yes`). |
| 6 | `balance` | Numerical (int) | Average yearly bank account balance recorded in Euros (€). |
| 7 | `housing` | Categorical (obj) | Housing loan flag: does the client hold an active housing mortgage? (`no`, `yes`). |
| 8 | `loan` | Categorical (obj) | Personal loan flag: does the client have an outstanding personal loan? (`no`, `yes`). |
| 9 | `contact` | Categorical (obj) | Contact communication medium (`cellular`, `telephone`, `unknown`). |
| 10 | `day` | Numerical (int) | Last contact calendar day of the month (1 through 31). |
| 11 | `month` | Categorical (obj) | Last contact month of the calendar year (`jan` through `dec`). |
| 12 | `duration` | Numerical (int) | Last contact call duration in seconds. *(Retained for EDA; excluded from predictive modeling to prevent leakage)*. |
| 13 | `campaign` | Numerical (int) | Number of marketing contacts performed during the current campaign for this client. |
| 14 | `pdays` | Numerical (int) | Days elapsed since the client was contacted from a previous campaign (-1 indicates client was not contacted previously). |
| 15 | `previous` | Numerical (int) | Number of contacts performed before the current campaign for this client. |
| 16 | `poutcome` | Categorical (obj) | Outcome of the previous marketing campaign (`failure`, `other`, `success`, `unknown`). |
| 17 | `deposit` | Categorical (obj) | **Target Variable**: Has the client subscribed to a term deposit? (`yes`, `no`). |


In [ ]:
# Structural Inspection and Data Types
print("=== DATASET STRUCTURAL INFO ===")
df.info()

print("\n=== DIMENSIONAL SUMMARY ===")
print(f"Total Rows:     {df.shape[0]:,}")
print(f"Total Columns:  {df.shape[1]}")
print(f"Memory Usage:   {df.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB")

# Identify Numerical vs Categorical Columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
cat_cols_features = [c for c in cat_cols if c != 'deposit']

print(f"\nNumerical Features ({len(num_cols)}):   {num_cols}")
print(f"Categorical Features ({len(cat_cols_features)}): {cat_cols_features}")
print(f"Target Column:               'deposit'")


In [ ]:
# Target Variable Distribution
target_counts = df['deposit'].value_counts()
target_pct = df['deposit'].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Frequency': target_counts,
    'Percentage (%)': target_pct.round(2)
})

print("=== TARGET VARIABLE DISTRIBUTION ===")
print(target_summary)


---
### 4. Data Cleaning

Data hygiene is fundamental to exploratory analysis and predictive modeling. In this section, we:
1. Standardize column identifiers (trim whitespace and cast to lowercase).
2. Clean object string columns of leading/trailing whitespace.
3. Investigate implicit `'unknown'` entries present in categorical columns.


In [ ]:
# Clean column identifiers and string entries
df.columns = [col.strip().lower() for col in df.columns]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Verify cleaning results
print("Columns verified clean. Current list of columns:")
print(df.columns.tolist())


---
### 5. Missing Value Analysis

We investigate two distinct categories of missingness:
1. **Explicit Missing Values**: Null, `NaN`, or empty entries checked via `df.isnull()`.
2. **Implicit / Hidden Missing Values**: Special domain sentinel values such as `'unknown'` strings in categorical variables, and `-1` in the `pdays` attribute (which signifies that the prospect was never previously contacted).


In [ ]:
# Explicit Missing Value Inspection
explicit_nulls = df.isnull().sum()
print("=== EXPLICIT MISSING VALUES (NaN) ===")
print(explicit_nulls[explicit_nulls > 0] if explicit_nulls.sum() > 0 else "Zero explicit NaN values detected across all columns.")

# Implicit Missing Value Inspection: 'unknown' strings in categorical attributes
unknown_counts = {}
for col in cat_cols_features:
    cnt = (df[col] == 'unknown').sum()
    if cnt > 0:
        unknown_counts[col] = {
            'Unknown Count': cnt,
            'Percentage (%)': round(cnt / len(df) * 100, 2)
        }

unknown_df = pd.DataFrame(unknown_counts).T
print("\n=== IMPLICIT MISSING ENTRIES ('unknown') ===")
print(unknown_df)

# pdays sentinel value (-1 represents uncontacted clients)
pdays_uncontacted = (df['pdays'] == -1).sum()
print(f"\nClients never previously contacted (pdays == -1): {pdays_uncontacted:,} ({pdays_uncontacted/len(df)*100:.2f}%)")


---
### 6. Duplicate Analysis

Duplicate records introduce artificial redundancy and bias statistical estimates. We execute an exhaustive duplicate search across all feature rows.


In [ ]:
# Duplicate Records Verification
num_duplicates = df.duplicated().sum()
print(f"Total Duplicate Rows Detected: {num_duplicates}")

if num_duplicates > 0:
    print(f"Dropping {num_duplicates} duplicate records...")
    df.drop_duplicates(inplace=True)
    print(f"New dataset shape: {df.shape}")
else:
    print("Dataset integrity confirmed: Zero duplicate records exist.")


---
### 7. Outlier Detection

Outliers in banking data can represent data collection anomalies or genuine high-net-worth clients and intense telemarketing runs. 

We apply the **Interquartile Range (IQR)** method:
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}, \quad \text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$

Records outside these bounds are quantified for each numerical attribute.


In [ ]:
# IQR-Based Outlier Detection
outlier_records = []

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_pct = (outlier_count / len(df)) * 100
    
    outlier_records.append({
        'Feature': col,
        'Q1 (25%)': round(q1, 2),
        'Q3 (75%)': round(q3, 2),
        'IQR': round(iqr, 2),
        'Lower Bound': round(lower_bound, 2),
        'Upper Bound': round(upper_bound, 2),
        'Outlier Count': outlier_count,
        'Outlier (%)': round(outlier_pct, 2)
    })

outlier_summary_df = pd.DataFrame(outlier_records)
print("=== NUMERICAL OUTLIER ANALYSIS (IQR METHOD) ===")
outlier_summary_df


---
### 8. Statistical Analysis

We conduct comprehensive descriptive statistical profiling across both numerical and categorical variables, examining central tendency (mean, median), dispersion (standard deviation, IQR), and distributional shape (skewness, kurtosis).


In [ ]:
# Comprehensive Numerical Descriptive Statistics
num_summary = df[num_cols].describe().T
num_summary['skewness'] = df[num_cols].skew()
num_summary['kurtosis'] = df[num_cols].kurt()
num_summary['median'] = df[num_cols].median()

reordered_cols = ['count', 'mean', 'std', 'skewness', 'kurtosis', 'min', '25%', 'median', '75%', 'max']
num_summary = num_summary[reordered_cols].round(2)

print("=== NUMERICAL DESCRIPTIVE STATISTICS ===")
num_summary


In [ ]:
# Categorical Descriptive Summary
cat_summary = df[cat_cols].describe().T
print("=== CATEGORICAL DESCRIPTIVE SUMMARY ===")
cat_summary


---
### 9. Univariate Analysis

Univariate analysis explores the individual distribution of key numerical and categorical variables in isolation.


In [ ]:
# Univariate Visualizations: Numerical Distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

univariate_num_features = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays']

for i, col in enumerate(univariate_num_features):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#1f77b4', bins=30)
    axes[i].set_title(f'Distribution of {col.capitalize()}', fontweight='bold')
    axes[i].set_xlabel(col.capitalize())
    axes[i].set_ylabel('Frequency')
    
plt.tight_layout()
plt.show()


In [ ]:
# Univariate Visualizations: Key Categorical Features
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Job Distribution
sns.countplot(data=df, y='job', order=df['job'].value_counts().index, ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title('Distribution of Occupations (Job)', fontweight='bold')
axes[0, 0].set_xlabel('Count')
axes[0, 0].set_ylabel('Job Type')

# Marital Status
sns.countplot(data=df, x='marital', order=df['marital'].value_counts().index, ax=axes[0, 1], palette='Purples_r')
axes[0, 1].set_title('Distribution of Marital Status', fontweight='bold')
axes[0, 1].set_xlabel('Marital Status')
axes[0, 1].set_ylabel('Count')

# Education Level
sns.countplot(data=df, x='education', order=df['education'].value_counts().index, ax=axes[1, 0], palette='Greens_r')
axes[1, 0].set_title('Distribution of Education Level', fontweight='bold')
axes[1, 0].set_xlabel('Education')
axes[1, 0].set_ylabel('Count')

# Target Variable Breakdown
deposit_counts = df['deposit'].value_counts()
axes[1, 1].pie(
    deposit_counts, 
    labels=[f"{k.capitalize()} ({v:,})" for k, v in deposit_counts.items()],
    autopct='%1.2f%%',
    startangle=140,
    colors=['#e74c3c', '#2ecc71'],
    explode=(0.04, 0.04),
    shadow=True
)
axes[1, 1].set_title('Target Distribution: Term Deposit Subscription', fontweight='bold')

plt.tight_layout()
plt.show()


---
### 10. Bivariate Analysis

Bivariate analysis explores the direct relationships between independent predictors and the target variable (`deposit`), testing for statistically significant differences between clients who subscribed versus those who declined.


In [ ]:
# Bivariate Analysis: Subscription Rate across Categorical Dimensions
from scipy.stats import chi2_contingency

categorical_test_cols = ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'poutcome']
print("=== CHI-SQUARE TEST OF INDEPENDENCE WITH TARGET ('deposit') ===")
print(f"{'Feature':<15} {'Chi2 Statistic':<16} {'p-value':<14} {'Statistical Significance'}")
print("-" * 65)

for col in categorical_test_cols:
    contingency = pd.crosstab(df[col], df['deposit'])
    chi2, p, dof, ex = chi2_contingency(contingency)
    sig = "Significant (p < 0.001)" if p < 0.001 else ("Significant (p < 0.05)" if p < 0.05 else "Not Significant")
    print(f"{col:<15} {chi2:<16.2f} {p:<14.2e} {sig}")


In [ ]:
# Subscription Percentages Across Key Categories
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Deposit by Housing Loan
housing_cross = pd.crosstab(df['housing'], df['deposit'], normalize='index') * 100
housing_cross[['yes', 'no']].plot(kind='bar', stacked=True, ax=axes[0, 0], color=['#2ecc71', '#e74c3c'])
axes[0, 0].set_title('Deposit Subscription by Housing Loan Status', fontweight='bold')
axes[0, 0].set_ylabel('Proportion (%)')
axes[0, 0].legend(['Subscribed (Yes)', 'Declined (No)'])
axes[0, 0].tick_params(axis='x', rotation=0)

# 2. Deposit by Personal Loan
loan_cross = pd.crosstab(df['loan'], df['deposit'], normalize='index') * 100
loan_cross[['yes', 'no']].plot(kind='bar', stacked=True, ax=axes[0, 1], color=['#2ecc71', '#e74c3c'])
axes[0, 1].set_title('Deposit Subscription by Personal Loan Status', fontweight='bold')
axes[0, 1].set_ylabel('Proportion (%)')
axes[0, 1].legend(['Subscribed (Yes)', 'Declined (No)'])
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. Deposit by Education Level
edu_cross = pd.crosstab(df['education'], df['deposit'], normalize='index') * 100
edu_cross[['yes', 'no']].plot(kind='bar', ax=axes[1, 0], color=['#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Subscription Rate by Education Level (%)', fontweight='bold')
axes[1, 0].set_ylabel('Percentage (%)')
axes[1, 0].tick_params(axis='x', rotation=25)

# 4. Deposit by Previous Campaign Outcome (poutcome)
poutcome_cross = pd.crosstab(df['poutcome'], df['deposit'], normalize='index') * 100
poutcome_cross[['yes', 'no']].plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'])
axes[1, 1].set_title('Subscription Rate by Previous Campaign Outcome (%)', fontweight='bold')
axes[1, 1].set_ylabel('Percentage (%)')
axes[1, 1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()


In [ ]:
# Bivariate Analysis: Continuous Predictors vs Deposit Status
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age vs Deposit
sns.boxplot(data=df, x='deposit', y='age', ax=axes[0], palette=['#e74c3c', '#2ecc71'])
axes[0].set_title('Age vs Deposit Subscription', fontweight='bold')
axes[0].set_xlabel('Deposit Subscribed')
axes[0].set_ylabel('Age')

# Balance vs Deposit (log scale for visual clarity due to skewness)
sns.boxplot(data=df, x='deposit', y='balance', ax=axes[1], palette=['#e74c3c', '#2ecc71'])
axes[1].set_yscale('symlog')
axes[1].set_title('Account Balance (Symlog Scale) vs Deposit', fontweight='bold')
axes[1].set_xlabel('Deposit Subscribed')
axes[1].set_ylabel('Balance (€)')

# Duration vs Deposit (Retained for EDA visualization only)
sns.boxplot(data=df, x='deposit', y='duration', ax=axes[2], palette=['#e74c3c', '#2ecc71'])
axes[2].set_title('Call Duration vs Deposit (EDA Insight Only)', fontweight='bold')
axes[2].set_xlabel('Deposit Subscribed')
axes[2].set_ylabel('Duration (Seconds)')

plt.tight_layout()
plt.show()


---
### 11. Multivariate Analysis

Multivariate analysis explores complex multi-feature interactions, pairwise collinearity, and high-dimensional clustering between demographic attributes, financial health, and campaign timing.


In [ ]:
# Numerical Correlation Heatmap
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, 
    mask=mask,
    annot=True, 
    fmt='.2f', 
    cmap='coolwarm', 
    vmin=-1, 
    vmax=1, 
    linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation Coefficient'}
)
plt.title('Correlation Heatmap of Numerical Features', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Multivariate Interaction: Age, Balance, and Deposit
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df[df['balance'] < 25000], 
    x='age', 
    y='balance', 
    hue='deposit', 
    alpha=0.45,
    palette={'no': '#e74c3c', 'yes': '#2ecc71'}
)
plt.title('Multivariate Interaction: Age vs Account Balance by Deposit Status', fontweight='bold')
plt.xlabel('Age (Years)')
plt.ylabel('Balance (€)')
plt.legend(title='Subscribed Deposit', loc='upper right')
plt.tight_layout()
plt.show()


---
### 12. Data Visualization (Exporting Publication-Quality Visuals)

As required by Assignment 2, we generate and save high-resolution (300 DPI) visualization figures directly into the `Visualizations/` directory as PNG files.


In [ ]:
# Visualization 1: Target Variable Distribution
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='deposit', palette=['#e74c3c', '#2ecc71'])
total_records = len(df)
for p in ax.patches:
    height = p.get_height()
    pct = (height / total_records) * 100
    ax.annotate(f'{height:,}\n({pct:.1f}%)', 
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', fontsize=12, color='white', fontweight='bold')

plt.title('Figure 1: Target Variable Distribution (Deposit Subscription)', fontweight='bold', fontsize=13)
plt.xlabel('Term Deposit Subscribed')
plt.ylabel('Client Count')
plt.tight_layout()
plt.savefig('Visualizations/01_target_distribution.png', dpi=300)
plt.show()
print("Saved: Visualizations/01_target_distribution.png")


In [ ]:
# Visualization 2: Age and Account Balance Distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age
sns.histplot(df['age'], kde=True, ax=axes[0], color='#2980b9', bins=35)
axes[0].axvline(df['age'].median(), color='red', linestyle='--', label=f"Median: {df['age'].median():.0f}")
axes[0].set_title('Age Distribution with Kernel Density Estimate', fontweight='bold')
axes[0].set_xlabel('Age (Years)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Balance (filtered for visual clarity)
sns.histplot(df[df['balance'].between(-2000, 15000)]['balance'], kde=True, ax=axes[1], color='#27ae60', bins=40)
axes[1].axvline(df['balance'].median(), color='red', linestyle='--', label=f"Median: €{df['balance'].median():.0f}")
axes[1].set_title('Account Balance Distribution (Clipped for View)', fontweight='bold')
axes[1].set_xlabel('Balance (€)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('Figure 2: Demographic and Financial Distributions', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('Visualizations/02_age_balance_distributions.png', dpi=300)
plt.show()
print("Saved: Visualizations/02_age_balance_distributions.png")


In [ ]:
# Visualization 3: Subscription Rate by Job Classification
job_rates = df.groupby('job')['deposit'].apply(lambda s: (s == 'yes').mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
bars = plt.barh(job_rates.index, job_rates.values, color='#3498db', edgecolor='black', alpha=0.85)
plt.axvline(x=target_pct['yes'], color='red', linestyle='--', linewidth=1.5, label=f'Baseline Rate ({target_pct["yes"]:.1f}%)')

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.8, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', 
             ha='left', va='center', fontsize=10, fontweight='bold')

plt.title('Figure 3: Term Deposit Subscription Rate Across Job Classifications', fontweight='bold', fontsize=13)
plt.xlabel('Subscription Rate (%)')
plt.ylabel('Job Classification')
plt.xlim(0, max(job_rates.values) + 8)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('Visualizations/03_subscription_rate_by_job.png', dpi=300)
plt.show()
print("Saved: Visualizations/03_subscription_rate_by_job.png")


In [ ]:
# Visualization 4: Subscription Rate by Housing and Personal Loan
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Housing Loan
housing_rates = df.groupby('housing')['deposit'].apply(lambda s: (s == 'yes').mean() * 100)
axes[0].bar(housing_rates.index, housing_rates.values, color=['#2ecc71', '#e74c3c'], edgecolor='black', alpha=0.85)
for i, val in enumerate(housing_rates.values):
    axes[0].text(i, val + 1.5, f'{val:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Subscription Rate by Housing Loan Status', fontweight='bold')
axes[0].set_xlabel('Housing Loan')
axes[0].set_ylabel('Subscription Rate (%)')
axes[0].set_ylim(0, 70)

# Personal Loan
loan_rates = df.groupby('loan')['deposit'].apply(lambda s: (s == 'yes').mean() * 100)
axes[1].bar(loan_rates.index, loan_rates.values, color=['#2ecc71', '#e74c3c'], edgecolor='black', alpha=0.85)
for i, val in enumerate(loan_rates.values):
    axes[1].text(i, val + 1.5, f'{val:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Subscription Rate by Personal Loan Status', fontweight='bold')
axes[1].set_xlabel('Personal Loan')
axes[1].set_ylabel('Subscription Rate (%)')
axes[1].set_ylim(0, 70)

plt.suptitle('Figure 4: Impact of Outstanding Liabilities on Subscription Propensity', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('Visualizations/04_subscription_by_housing_and_loan.png', dpi=300)
plt.show()
print("Saved: Visualizations/04_subscription_by_housing_and_loan.png")


In [ ]:
# Visualization 5: Impact of Previous Campaign Outcome (poutcome)
poutcome_rates = df.groupby('poutcome')['deposit'].apply(lambda s: (s == 'yes').mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
bars = plt.bar(poutcome_rates.index, poutcome_rates.values, color=['#27ae60', '#f39c12', '#95a5a6', '#e74c3c'], edgecolor='black')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 1.5, f'{height:.1f}%', ha='center', fontweight='bold')

plt.title('Figure 5: Subscription Conversion Rate by Previous Campaign Outcome (Poutcome)', fontweight='bold', fontsize=13)
plt.xlabel('Previous Campaign Outcome')
plt.ylabel('Deposit Subscription Rate (%)')
plt.ylim(0, 105)
plt.tight_layout()
plt.savefig('Visualizations/05_poutcome_vs_deposit.png', dpi=300)
plt.show()
print("Saved: Visualizations/05_poutcome_vs_deposit.png")


In [ ]:
# Visualization 6: Call Duration Distribution by Deposit Status (EDA Only)
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x='duration', hue='deposit', common_norm=False, fill=True, palette={'no': '#e74c3c', 'yes': '#2ecc71'}, alpha=0.4)
plt.xlim(0, 2000)
plt.title('Figure 6: Contact Duration Distribution by Subscription Status (EDA Diagnostic)', fontweight='bold', fontsize=13)
plt.xlabel('Call Duration (Seconds)')
plt.ylabel('Density')
plt.annotate('Data Leakage Warning:\nCall duration cannot be known\nprior to the call. Excluded\nfrom predictive modeling.',
             xy=(1200, 0.0010), xytext=(1300, 0.0016),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6),
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff2ac', alpha=0.8),
             fontsize=10)
plt.tight_layout()
plt.savefig('Visualizations/06_duration_vs_deposit_eda.png', dpi=300)
plt.show()
print("Saved: Visualizations/06_duration_vs_deposit_eda.png")


In [ ]:
# Visualization 7: Correlation Matrix Heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Figure 7: Pearson Correlation Matrix of Numerical Features', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('Visualizations/07_correlation_matrix.png', dpi=300)
plt.show()
print("Saved: Visualizations/07_correlation_matrix.png")


---
### 13. Data Preprocessing

#### 13.1 Rigorous Data Leakage Prevention
In strict compliance with Assignment 2 instructions and the UCI Machine Learning benchmark rules:
> **CRITICAL**: The attribute `duration` heavily influences the target output (e.g., if duration=0 then $y='no'$). Yet, the duration of a call is intrinsically unknown prior to making the call. To construct a realistic, prospective predictive model that can guide telemarketing outreach **before** dialing, `duration` is **strictly excluded** from feature matrix $X$.


In [ ]:
# Data Leakage Prevention: Drop 'duration' from Feature Matrix
print("Executing Data Leakage Prevention Protocol...")
print(f"Features before dropping: {df.columns.tolist()}")

# Drop target and duration from predictor set
X_raw = df.drop(columns=['deposit', 'duration'])
y = (df['deposit'] == 'yes').astype(int).values

print(f"Features retained for modeling ({X_raw.shape[1]}): {X_raw.columns.tolist()}")
print(f"Target vector shape: {y.shape} (Positive Class Rate: {y.mean()*100:.2f}%)")


In [ ]:
# Categorical Feature Encoding (One-Hot Encoding)
modeling_cat_cols = X_raw.select_dtypes(include=['object']).columns.tolist()
modeling_num_cols = X_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical features to encode ({len(modeling_cat_cols)}): {modeling_cat_cols}")
print(f"Numerical features to scale   ({len(modeling_num_cols)}): {modeling_num_cols}")

# One-hot encode categorical features with drop_first=True to avoid collinearity
X_encoded = pd.get_dummies(X_raw, columns=modeling_cat_cols, drop_first=True, dtype=float)

print(f"\nTransformed Feature Space: {X_encoded.shape[1]} dimensions (from {X_raw.shape[1]} original features).")
feature_names = X_encoded.columns.tolist()
print(f"Sample Encoded Column Headers:\n{feature_names[:10]} ...")


---
### 14. Train, Validation and Test Split

To properly train, tune, and evaluate our Deep Learning Artificial Neural Network without data leakage:
- **Training Set (70%)**: Used by backpropagation to optimize neural network weights.
- **Validation Set (15%)**: Used by Early Stopping and Learning Rate schedules to monitor generalization during training.
- **Test Set (15%)**: Completely held out until final model assessment.

We employ **stratified splitting** to maintain exact class proportions across all three partitions, and fit the `StandardScaler` strictly on the training partition.


In [ ]:
# Stratified 70% Train, 15% Validation, 15% Test Split
X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X_encoded, y, 
    test_size=0.30, 
    random_state=SEED, 
    stratify=y
)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50, 
    random_state=SEED, 
    stratify=y_temp
)

# Feature Scaling (Fit on Train ONLY to prevent data snooping)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

# Verify split dimensions and class balance
print("=== PARTITION SUMMARY & CLASS STRATIFICATION ===")
for name, partition_y, shape in [
    ('Training Set', y_train, X_train.shape),
    ('Validation Set', y_val, X_val.shape),
    ('Test Set', y_test, X_test.shape)
]:
    pos_rate = partition_y.mean() * 100
    print(f"{name:<16}: {shape[0]:,} samples ({shape[0]/len(y)*100:.1f}%) | Positive Class: {pos_rate:.2f}% | Shape: {shape}")


---
### 15. Deep Learning using TensorFlow/Keras

We construct an **Artificial Neural Network (ANN)** for binary classification. 

#### Network Architecture:
1. **Input Layer**: Dimension = $41$ input features.
2. **Dense Hidden Layer 1**: $64$ neurons, ReLU activation function, $L_2$ weight regularization ($\lambda = 10^{-4}$).
3. **Batch Normalization 1**: Stabilizes gradient propagation and speeds up convergence.
4. **Dropout Layer 1**: Rate = $0.30$ to prevent co-adaptation of hidden units.
5. **Dense Hidden Layer 2**: $32$ neurons, ReLU activation, $L_2$ weight regularization.
6. **Batch Normalization 2**.
7. **Dropout Layer 2**: Rate = $0.20$.
8. **Dense Hidden Layer 3**: $16$ neurons, ReLU activation.
9. **Output Layer**: $1$ neuron with Sigmoid activation function $\sigma(z) = \frac{1}{1 + e^{-z}}$ outputting the probability of deposit subscription.

#### Compilation & Training Optimization:
- **Optimizer**: Adam with learning rate $\alpha = 0.001$.
- **Loss**: Binary Cross-Entropy (Log Loss).
- **Callbacks**:
  - `EarlyStopping`: Monitors `val_loss`, patience = 12 epochs, automatically restores the best network weights.
  - `ReduceLROnPlateau`: Halves the learning rate if `val_loss` plateaus for 5 consecutive epochs.


In [ ]:
# Define the Neural Network Architecture
def build_ann_model(input_dim):
    model = keras.Sequential([
        # Input Layer
        layers.Input(shape=(input_dim,)),
        
        # Dense Block 1
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4), name='dense_1'),
        layers.BatchNormalization(name='batchnorm_1'),
        layers.Dropout(0.30, name='dropout_1'),
        
        # Dense Block 2
        layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4), name='dense_2'),
        layers.BatchNormalization(name='batchnorm_2'),
        layers.Dropout(0.20, name='dropout_2'),
        
        # Dense Block 3
        layers.Dense(16, activation='relu', name='dense_3'),
        
        # Output Layer (Binary Classification via Sigmoid)
        layers.Dense(1, activation='sigmoid', name='output_layer')
    ], name='Bank_Marketing_ANN')
    
    # Compile Model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )
    return model

model = build_ann_model(X_train.shape[1])
model.summary()


In [ ]:
# Define Training Callbacks
model_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1
    )
]

# Train the Deep Learning Model
print("Initiating Deep Learning Model Training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=64,
    callbacks=model_callbacks,
    verbose=1
)
print("Model training successfully completed!")


---
### 16. Model Evaluation

We evaluate the trained Artificial Neural Network on the completely unseen **Test Set** ($N \approx 1,675$ records).

We produce and save:
1. **Confusion Matrix Heatmap** (`Visualizations/08_confusion_matrix.png`)
2. **ROC Curve** (`Visualizations/09_roc_curve.png`)
3. **Training & Validation Loss Curve** (`Visualizations/10_loss_curve.png`)
4. **Training & Validation Accuracy Curve** (`Visualizations/11_accuracy_curve.png`)


In [ ]:
# Predict on Unseen Test Partition
y_pred_probs = model.predict(X_test, verbose=0).ravel()
y_pred_classes = (y_pred_probs >= 0.50).astype(int)

# Compute Core Performance Metrics
test_loss, test_acc, test_auc, test_prec, test_rec = model.evaluate(X_test, y_test, verbose=0)
test_f1 = f1_score(y_test, y_pred_classes)

print("=" * 45)
print("     UNSEEN TEST SET EVALUATION METRICS     ")
print("=" * 45)
print(f"Test Loss:             {test_loss:.4f}")
print(f"Test Accuracy:         {test_acc*100:.2f}%")
print(f"Test ROC-AUC Score:    {test_auc:.4f}")
print(f"Test Precision:        {test_prec*100:.2f}%")
print(f"Test Recall:           {test_rec*100:.2f}%")
print(f"Test F1-Score:         {test_f1:.4f}")
print("=" * 45)

# Detailed Classification Report
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred_classes, target_names=['Decline (No)', 'Subscribe (Yes)']))


In [ ]:
# Evaluation Visualization 1: Confusion Matrix
cm = confusion_matrix(y_test, y_pred_classes)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    cbar=False,
    xticklabels=['Predicted No', 'Predicted Yes'],
    yticklabels=['Actual No', 'Actual Yes'],
    annot_kws={'size': 14, 'weight': 'bold'},
    ax=ax
)

# Overlay normalized percentages
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j + 0.5, i + 0.7, f"({cm_norm[i, j]*100:.1f}%)", 
                ha='center', va='center', color='gray' if cm[i, j] < cm.max()/2 else 'white', fontsize=11)

plt.title('Figure 8: Test Set Confusion Matrix', fontweight='bold', fontsize=13)
plt.ylabel('Actual Ground Truth')
plt.xlabel('Neural Network Prediction')
plt.tight_layout()
plt.savefig('Visualizations/08_confusion_matrix.png', dpi=300)
plt.show()
print("Saved: Visualizations/08_confusion_matrix.png")


In [ ]:
# Evaluation Visualization 2: ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_probs)
roc_auc_val = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#2980b9', lw=2.5, label=f'ANN Classifier (AUC = {roc_auc_val:.4f})')
plt.plot([0, 1], [0, 1], color='#7f8c8d', lw=1.5, linestyle='--', label='Random Chance (AUC = 0.5000)')
plt.xlim([-0.02, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=11)
plt.title('Figure 9: Receiver Operating Characteristic (ROC) Curve', fontweight='bold', fontsize=13)
plt.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig('Visualizations/09_roc_curve.png', dpi=300)
plt.show()
print("Saved: Visualizations/09_roc_curve.png")


In [ ]:
# Evaluation Visualizations 3 & 4: Loss and Accuracy Learning Curves
epochs_range = range(1, len(history.history['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss Curve
axes[0].plot(epochs_range, history.history['loss'], label='Training Loss', color='#e74c3c', lw=2)
axes[0].plot(epochs_range, history.history['val_loss'], label='Validation Loss', color='#3498db', lw=2, linestyle='--')
axes[0].set_title('Figure 10: Training vs. Validation Loss Curve', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].legend()

# Accuracy Curve
axes[1].plot(epochs_range, history.history['accuracy'], label='Training Accuracy', color='#2ecc71', lw=2)
axes[1].plot(epochs_range, history.history['val_accuracy'], label='Validation Accuracy', color='#9b59b6', lw=2, linestyle='--')
axes[1].set_title('Figure 11: Training vs. Validation Accuracy Curve', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('Visualizations/10_loss_curve.png', dpi=300)
fig.savefig('Visualizations/11_accuracy_curve.png', dpi=300)
plt.show()
print("Saved: Visualizations/10_loss_curve.png")
print("Saved: Visualizations/11_accuracy_curve.png")


---
### 17. Feature Importance

Because neural networks are non-linear models, we apply **Permutation Feature Importance** on the test partition.

Permutation importance shuffles each feature column one at a time and records the resulting degradation in test ROC-AUC. Features that produce the largest drop in ROC-AUC when randomized are the most critical predictors.


In [ ]:
# Model-Agnostic Permutation Feature Importance
baseline_auc = roc_auc_score(y_test, y_pred_probs)
importance_scores = {}

# Evaluate feature importance by permuting each feature on the test set
for idx, col_name in enumerate(feature_names):
    X_test_permuted = X_test.copy()
    # Shuffle the specific feature column
    np.random.seed(SEED)
    shuffled_col = np.random.permutation(X_test_permuted[:, idx])
    X_test_permuted[:, idx] = shuffled_col
    
    # Predict with permuted column
    perm_probs = model.predict(X_test_permuted, verbose=0).ravel()
    perm_auc = roc_auc_score(y_test, perm_probs)
    
    # Drop in AUC is the importance score
    importance_scores[col_name] = max(0, baseline_auc - perm_auc)

importance_df = pd.DataFrame(list(importance_scores.items()), columns=['Feature', 'Importance_Drop_in_AUC'])
importance_df = importance_df.sort_values(by='Importance_Drop_in_AUC', ascending=False).reset_index(drop=True)

print("=== TOP 15 MOST INFLUENTIAL PREDICTORS ===")
importance_df.head(15)


In [ ]:
# Feature Importance Visualization
top_n = 12
top_features = importance_df.head(top_n).sort_values(by='Importance_Drop_in_AUC', ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(top_features['Feature'], top_features['Importance_Drop_in_AUC'], color='#16a085', edgecolor='black')
plt.title(f'Figure 12: Top {top_n} Features by Permutation Importance (AUC Drop)', fontweight='bold', fontsize=13)
plt.xlabel('Decrease in Test ROC-AUC Upon Permutation')
plt.ylabel('Feature Name')

for bar in bars:
    width = bar.get_width()
    if width > 0.001:
        plt.text(width + 0.0005, bar.get_y() + bar.get_height()/2, f'{width:.4f}', 
                 ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('Visualizations/12_feature_importance.png', dpi=300)
plt.show()
print("Saved: Visualizations/12_feature_importance.png")


---
### 18. Conclusion

#### 18.1 Summary of Exploratory Data Analysis Insights
1. **Prior Campaign Success (`poutcome == 'success'`) is the Single Strongest Predictor**: Clients whose previous marketing interaction resulted in a successful subscription have a staggering **91.3%** conversion rate in the current campaign. Prior positive bank relationship is the primary conversion asset.
2. **Liabilities Inversely Impact Savings Capacity**: Clients with active housing loans convert at only **36.6%** compared to **57.0%** for clients without housing mortgages. Similarly, personal loan holders convert at only **33.2%**. Debt-free clients retain substantially higher discretionary cash flow to lock into term deposits.
3. **Financial Capital Matters**: Clients who subscribed had a higher average account balance (€1,804 vs €1,280).
4. **Occupational Segments**: Students (68.4%) and retirees (65.5%) demonstrate the highest propensity to subscribe, whereas blue-collar workers (36.4%) exhibit the lowest conversion rate.
5. **Marketing Campaign Fatigue**: Repeated contacts within a single campaign degrade conversion probability; prospects contacted more than 3 to 4 times show diminishing returns.

#### 18.2 Deep Learning Modeling Without Data Leakage
- By deliberately **excising call duration** prior to neural network training, we eliminated target leakage, ensuring the model functions as a genuine prospective predictive scoring engine.
- The TensorFlow/Keras Artificial Neural Network (incorporating Batch Normalization, Dropout, and $L_2$ regularization) demonstrated strong generalization on the test partition without overfitting, achieving balanced precision and recall.

#### 18.3 Actionable Strategic Recommendations for Banking Operations
1. **Target Repeat Prospects First**: Prioritize contact lists for individuals with `poutcome == 'success'`.
2. **Pre-screen for Debt Obligations**: Exclude or de-prioritize customers carrying heavy mortgage or personal loans in mass deposit drives.
3. **Tailor Outreach to Students and Retirees**: Formulate age-bracketed deposit incentive programs for students and pension-age individuals.
4. **Cap Campaign Intensity**: Establish a maximum contact cap (3 attempts) per prospect to mitigate customer frustration and reduce telemarketing expenditure.


---
### 19. References

1. **Moro, S., Cortez, P., & Rita, P. (2014)**. *A data-driven approach to predict the success of bank telemarketing.* Decision Support Systems, 62, 22-31. Elsevier. DOI: [10.1016/j.dss.2014.03.001](https://doi.org/10.1016/j.dss.2014.03.001).
2. **UCI Machine Learning Repository**: *Bank Marketing Dataset*. Available at: [https://archive.ics.uci.edu/ml/datasets/Bank+Marketing](https://archive.ics.uci.edu/ml/datasets/Bank+Marketing).
3. **Abadi, M., et al. (2016)**. *TensorFlow: A system for large-scale machine learning.* In 12th USENIX Symposium on Operating Systems Design and Implementation (OSDI 16), 265-283.
4. **Chollet, F. (2021)**. *Deep Learning with Python (2nd ed.)*. Manning Publications.
5. **Pedregosa, F., et al. (2011)**. *Scikit-learn: Machine Learning in Python.* Journal of Machine Learning Research, 12, 2825-2830.
6. **Waskom, M. L. (2021)**. *Seaborn: statistical data visualization.* Journal of Open Source Software, 6(60), 3021.
7. **Hunter, J. D. (2007)**. *Matplotlib: A 2D graphics environment.* Computing in Science & Engineering, 9(3), 90-95.
